In [7]:
###########################################
## Model 1 : Baseline - Exact DP ##
###########################################

import numpy as np
import random

### Exact DP

# Parameter
ages = list(range(18, 26))  # age from 18 to 25
educ_levels = [0, 1, 2, 3, 4]
asset_levels = [-10000, -5000, 0, 5000, 10000]
abilities = ['low', 'medium', 'high']
ability_map = {'low': 0, 'medium': 1, 'high': 2}
delta = 0.95
tuition = 3000
wage_base = {0: 6000, 1: 10000, 2: 15000, 3: 20000, 4: 25000}
wage_shocks = [-3000, 0, 3000]
shock_probs = [0.25, 0.5, 0.25]
a_min = min(asset_levels)
rho = 0.95  # risk aversion

ability_wage_bonus = {'low': -500, 'medium': 0, 'high': 1000}

# CRRA utility
def crra(c, rho):
    return np.sign(c) * (abs(c) ** (1 - rho)) / (1 - rho)

def behavior_utility(action):
    return {'study': -5, 'work': -10, 'delay': -2}[action]

def prob_educ_success(ability, educ):
    p = np.array([[0.7, 0.7, 0.6, 0.5, 0.5],
                  [0.8, 0.7, 0.6, 0.6, 0.6],
                  [0.9, 0.8, 0.7, 0.7, 0.7]])
    return p[ability_map[ability]][educ]

def expected_wage(educ, ability):
    return wage_base[educ] + ability_wage_bonus[ability]

# Initialize the state space
V = {}
policy = {}
states = []
for age in ages:
    for educ in educ_levels:
        if age == 18 and educ > 0:
            continue
        if educ > (age - 18):
            continue
        for asset in asset_levels:
            for ability in abilities:
                state = (age, educ, asset, ability)
                states.append(state)
                V[state] = 0
                policy[state] = None

def terminal_reward(educ, asset, rho):
    asset_consumption = max(asset, 1)
    return crra(asset_consumption, rho) + 0.5 * educ

# Terminal reward
for state in states:
    age, educ, asset, ability = state
    if age == 25:
        V[state] = terminal_reward(educ, asset, rho=rho)

def expected_reward(state, action, educ_next, rho=1.5):
    age, educ, asset_now, ability = state
    act, asset_next = action
    if asset_next < a_min :
        return -np.inf
    base_income = 0
    cost = 0
    if act == 'study':
        cost = tuition
    elif act == 'work':
        base_income = expected_wage(educ, ability)
    reward_sum = 0.0
    for shock, prob in zip(wage_shocks, shock_probs):
        income = base_income + shock if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            return -np.inf
        utility = crra(consumption, rho)
        if utility == -np.inf:
            return -np.inf
        utility += behavior_utility(act)
        reward_sum += prob * utility
    return reward_sum

# Transition of state variables
def deterministic_transitions(state, action):
    age, educ, asset_now, ability = state
    act, asset_next = action
    next_age = age + 1
    outcomes = []
    if act == 'study':
        p_succ = prob_educ_success(ability, educ)
        educ_outcomes = [(min(educ + 1, 4), p_succ), (educ, 1 - p_succ)]
    else:
        educ_outcomes = [(educ, 1.0)]
    for educ_next, p_educ in educ_outcomes:
        for next_ability, p_ability in ability_transition[ability].items():
            next_state = (next_age, educ_next, asset_next, next_ability)
            prob = p_educ * p_ability
            outcomes.append((next_state, prob))
    return outcomes

ability_transition = {
    'low':    {'low': 0.5, 'medium': 0.4, 'high': 0.1},
    'medium': {'low': 0.1, 'medium': 0.6, 'high': 0.3},
    'high':   {'low': 0.0, 'medium': 0.2, 'high': 0.8}
}

action_types = ['study', 'work', 'delay']
ACTIONS = [(act, a_next) for act in action_types for a_next in asset_levels if a_next >= a_min]

trajectory_debug = []

# Backward value iteration
for t in reversed(ages[:-1]):
    for state in states:
        age, educ, asset, ability = state
        if age != t:
            continue
        best_value = -np.inf
        best_action = None
        best_debug = ""
        for action in ACTIONS:
            total = 0.0
            transitions = deterministic_transitions(state, action)
            if not transitions:
                continue
            skip = False
            debug = []
            for next_state, prob in transitions:
                if next_state[0] > 25:
                    continue
                rew = expected_reward(state, action, next_state[1], rho=rho)
                if rew == -np.inf:
                    skip = True
                    break
                val = rew + delta * V[next_state]
                debug.append((next_state, prob, rew, V[next_state], val))
                total += prob * val
            if skip:
                continue
            if total > best_value:
                best_value = total
                best_action = action
                best_debug = debug
        V[state] = best_value
        policy[state] = best_action
        if best_action is not None and state[0] >= 18:
            trajectory_debug.append((state, best_action, best_value, best_debug))


# Show the policy trajectory
print("\nExact policy trajectory from state (18, 0, 0, 'high'):")

state = (18, 0, 0, 'high')
for t in range(18, 25):
    action = policy.get(state, None)
    value = V.get(state, None)
    if action is None or value is None:
        print(f"Age {t}: No valid action or value found for state {state}")
        break

    print(f"Age {t}: State = {state}, V* = {value:.3f}, Optimal Action = {action}")

    next_states = deterministic_transitions(state, action)
    print(f"        Possible next states:")
    for ns, prob in next_states:
        print(f"          -> {ns} with probability {prob:.2f}")

    next_states_sorted = sorted(next_states, key=lambda x: -x[1])
    state = next_states_sorted[0][0]



Exact policy trajectory from state (18, 0, 0, 'high'):
Age 18: State = (18, 0, 0, 'high'), V* = 165.781, Optimal Action = ('study', -5000)
        Possible next states:
          -> (19, 1, -5000, 'low') with probability 0.00
          -> (19, 1, -5000, 'medium') with probability 0.18
          -> (19, 1, -5000, 'high') with probability 0.72
          -> (19, 0, -5000, 'low') with probability 0.00
          -> (19, 0, -5000, 'medium') with probability 0.02
          -> (19, 0, -5000, 'high') with probability 0.08
Age 19: State = (19, 1, -5000, 'high'), V* = 149.889, Optimal Action = ('study', -10000)
        Possible next states:
          -> (20, 2, -10000, 'low') with probability 0.00
          -> (20, 2, -10000, 'medium') with probability 0.16
          -> (20, 2, -10000, 'high') with probability 0.64
          -> (20, 1, -10000, 'low') with probability 0.00
          -> (20, 1, -10000, 'medium') with probability 0.04
          -> (20, 1, -10000, 'high') with probability 0.16
Age 2

In [13]:
#################################################
## Model 1 : Baseline - Approximate DP ##
#################################################

# The reasons for feature vector processing are as follows:
# - Structural unification: scaling and integrating different types and meanings of state and action information into a standard numerical vector.
# - Input preparation: providing a standard, numerical input for the subsequent function approximator.
def feature_vector(state, action):
    age, educ, asset, ability = state
    act, asset_next = action
    ability_idx = ability_map[ability]
    return np.array([
        age - 18, educ, asset/10000, ability_idx,
        asset_next, action_types.index(act)
    ], dtype=float)

# The "score" of each action is evaluated by calculating the dot product of the parameter vector θ and the state-action feature vector ϕ(s,a),
# and then the Softmax function is used to convert these scores into the probability of taking each action a in state s. The Policy Gradient
# algorithm will update the parameters θ based on the collected states, actions, and rewards, thereby adjusting these probability distributions
# so that actions that can obtain higher total rewards have a greater probability of being selected.
def softmax_policy(state, theta, actions=ACTIONS):
    logits = np.array([np.dot(theta, feature_vector(state, a)) for a in actions])
    exps = np.exp(logits - np.max(logits))
    return exps / np.sum(exps)

# Legal actions setup purpose:
# - We want to first call legal_actions(state) to get a list of legal actions valid_actions.
# - Then calculate np.dot(theta, feature_vector(state, a)) only for the actions in valid_actions.
# - Finally, apply Softmax on the scores of these legal actions to get the probability distribution only for legal actions.
def legal_actions(state):
    age, educ, asset_now, ability = state
    legal = []
    for action in ACTIONS:
        act, asset_next = action
        if asset_next < a_min:
            continue
        base_income = expected_wage(educ, ability) if act == 'work' else 0
        cost = tuition if act == 'study' else 0
        # worst-case shock
        worst_shock = min(wage_shocks)
        income = base_income + worst_shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            continue
        legal.append(action)
    return legal

# The agent interacts with the environment according to the action chosen by the choose_action function,
# obtaining a reward and the next state, thus generating a complete state-action-reward sequence (trajectory).
# These trajectory data are then used to calculate the gradient and update the policy parameters θ.
def choose_action(state, theta):
    legal = legal_actions(state)
    if not legal:
        return None
    probs = softmax_policy(state, theta, actions=legal)
    return legal[np.random.choice(len(legal), p=probs)]

# Record the agent's decisions and the environment's feedback to provide data for strategy learning.
def simulate_trajectory(s0, theta):
    trajectory = []
    state = s0
    for t in range(18, 25):
        action = choose_action(state, theta)
        age, educ, asset_now, ability = state
        act, asset_next = action

        # illegal actions
        if asset_next < a_min :
            break

        base_income = 0
        cost = 0
        if act == 'study':
            cost = tuition
        elif act == 'work':
            base_income = expected_wage(educ, ability)

        shock = random.choices(wage_shocks, weights=shock_probs)[0] if act == 'work' else 0
        income = base_income + shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            break

        reward = crra(consumption, rho) + behavior_utility(act)

        # next state
        if act == 'study':
            p_succ = prob_educ_success(ability, educ)
            educ_next = educ + 1 if random.random() < p_succ else educ
        else:
            educ_next = educ
        educ_next = min(educ_next, 4)

        next_ability = random.choices(
            list(ability_transition[ability].keys()),
            weights=list(ability_transition[ability].values())
        )[0]
        next_state = (age + 1, educ_next, asset_next, next_ability)

        trajectory.append((state, action, reward))
        state = next_state

    # terminal reward
    final_rew = terminal_reward(state[1], state[2], rho=rho)
    trajectory.append((state, None, final_rew))
    return trajectory

# Use backward recursion to calculate the total discounted reward starting from each time step in the trajectory.
def compute_returns(traj, gamma=delta):
    G = 0
    returns = []
    for _, _, r in reversed(traj):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

# Update parameters using gradient ascent
def compute_policy_gradient(traj, returns, theta):
    grads = np.zeros_like(theta)
    for (s, a, _), G in zip(traj, returns):
        if a is None:
            continue
        probs = softmax_policy(s, theta)
        phi = np.array([feature_vector(s, act) for act in ACTIONS])
        grad_logpi = feature_vector(s, a) - np.dot(probs, phi)
        grads += G * grad_logpi
    return grads

# Continuous iteration, the strategy parameters are optimized in the direction of obtaining higher expected returns.
def train_policy_gradient(theta_init, alpha=0.01, epochs=200, episodes_per_epoch=50):
    theta = theta_init.copy()
    history = []
    for epoch in range(epochs):
        grads_all = []
        returns_all = []
        for _ in range(episodes_per_epoch):
            s0 = (18, 0, 0, 'high')
            traj = simulate_trajectory(s0, theta)
            R = compute_returns(traj)
            grad = compute_policy_gradient(traj, R, theta)
            grads_all.append(grad)
            returns_all.append(R[0])  # return at start
        theta += alpha * np.mean(grads_all, axis=0)
        avg_return = np.mean(returns_all)
        history.append(avg_return)
        if epoch % 10 == 0:
            print(f"[Epoch {epoch}] Average Return = {avg_return:.2f}")
    return theta, history

np.random.seed(42)
theta0 = np.random.randn(6)
theta_trained, return_history = train_policy_gradient(theta0)

s0 = (18, 0, 0, 'high')
vals = [compute_returns(simulate_trajectory(s0, theta_trained))[0] for _ in range(200)]
print(f"Estimated V(s0) ≈ {np.mean(vals):.2f}")

# "choose_action" with determinstic environment to simplify the results presentation
def print_policy_trajectory(theta, start_state=s0):
    print("\n🧭 Policy Trajectory from state", start_state)
    state = start_state
    for age in range(18, 25):
        action = choose_action(state, theta)
        if action is None:
            print(f"Age {age}: State = {state} → ❌ No Action Available")
            break

        print(f"Age {age}: State = {state}, ➡️ Chosen Action = {action}")

        age, educ, asset_now, ability = state
        act, asset_next = action

        if act == 'work':
            base_income = expected_wage(educ, ability)
        else:
            base_income = 0
        cost = tuition if act == 'study' else 0
        income = base_income

        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            print(f"⚠️ Consumption <= 0, terminate.")
            break

        if act == 'study':
            p_succ = prob_educ_success(ability, educ)
            if p_succ >= 0.5:
                educ_next = min(4, educ + 1)
            else:
                educ_next = educ
        else:
            educ_next = educ

        next_ability = max(
            ability_transition[ability].items(),
            key=lambda x: x[1]
        )[0]

        next_state = (age + 1, educ_next, asset_next, next_ability)
        state = next_state

        if state[0] >= 25:
            break

print_policy_trajectory(theta_trained)


[Epoch 0] Average Return = 147.16
[Epoch 10] Average Return = 148.76
[Epoch 20] Average Return = 148.44
[Epoch 30] Average Return = 148.63
[Epoch 40] Average Return = 148.92
[Epoch 50] Average Return = 149.02
[Epoch 60] Average Return = 149.07
[Epoch 70] Average Return = 149.33
[Epoch 80] Average Return = 148.54
[Epoch 90] Average Return = 148.55
[Epoch 100] Average Return = 149.51
[Epoch 110] Average Return = 148.12
[Epoch 120] Average Return = 148.97
[Epoch 130] Average Return = 148.79
[Epoch 140] Average Return = 149.09
[Epoch 150] Average Return = 148.82
[Epoch 160] Average Return = 149.04
[Epoch 170] Average Return = 148.60
[Epoch 180] Average Return = 148.70
[Epoch 190] Average Return = 149.26
Estimated V(s0) ≈ 149.08

🧭 Policy Trajectory from state (18, 0, 0, 'high')
Age 18: State = (18, 0, 0, 'high'), ➡️ Chosen Action = ('study', -10000)
Age 19: State = (19, 1, -10000, 'high'), ➡️ Chosen Action = ('work', -10000)
Age 20: State = (20, 1, -10000, 'high'), ➡️ Chosen Action = ('wor